# Effective resistance vs. film thickness d (3D, distance-dependent model)

`effective_resistance_vs_N_a.ipynb` studies $R_{eff}(N, a)$ in 2D, and `percolation_3d.ipynb` moves the pipeline to the unit cube with a contact-distance bonding criterion. A real film has a large footprint and a small thickness, so this notebook adds the third variable: **how does $R_{eff}(N, a, d)$ depend on the thickness d at fixed N and a?**

`point_generator_3d.generate_points_3d` and `pipeline_utils_3d.generate_percolating_network_3d` accept `box_dims=(1.0, 1.0, d)`, which confines the sticks to the slab [0, 1] × [0, 1] × [0, d]. The default `(1.0, 1.0, 1.0)` is the unit cube, so the other 3D notebooks are unaffected. Transport stays along x, with source and drain faces at x < 0.1 and x > 0.9.

**Modeling choice.** N is held fixed while d varies, so a thinner film packs the same material into less volume. Fixed areal density (N proportional to d) is a different, coupled question; see Section 10.

---
## 1. Setup and imports

In [1]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    utils_path = "/content/drive/MyDrive/ColabProject/utils"
    print("Environment: Google Colab")
except ImportError:
    utils_path = os.path.abspath(os.path.join(os.getcwd(), "..", "utils"))
    print("Environment: Local")

if utils_path not in sys.path:
    sys.path.append(utils_path)

print(f"Utils path: {utils_path}")


Environment: Local
Utils path: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils


In [2]:
from pipeline_utils_3d import generate_percolating_network_3d
from pipeline_utils import distance_dependent_impedance, stable_seed
from network_visualization_3d import plot_network_3d_colored_by_resistance
from point_generator_3d import generate_points_3d

print("Imports OK")


Imports OK


---
## 2. Experiment tracking

Figures and sweep data go to a new `experiments/exp_N/` folder created by `ExperimentManager`.

In [3]:
from experiment_manager import setup_experiment

exp_manager = setup_experiment()


EXPERIMENT CONFIGURATION
  Environment: Local
  Experiment #: 872
  Timestamp: 20260915_202740
  Results directory: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_872



---
## 3. Sanity check: sticks stay inside the slab

Generate sticks in a thin slab and check that every z-coordinate lies in [0, d] while x and y span [0, 1].

In [4]:
d_demo = 0.1
p1_demo, p2_demo, dir_demo = [], [], []
np.random.seed(0)
generate_points_3d(150, p1_demo, p2_demo, dir_demo, a=0.3, box_dims=(1.0, 1.0, d_demo))

all_pts = np.array(p1_demo + p2_demo)
print(f"box_dims = (1.0, 1.0, {d_demo})")
print(f"  x range: [{all_pts[:,0].min():.4f}, {all_pts[:,0].max():.4f}]  (expected within [0, 1])")
print(f"  y range: [{all_pts[:,1].min():.4f}, {all_pts[:,1].max():.4f}]  (expected within [0, 1])")
print(f"  z range: [{all_pts[:,2].min():.4f}, {all_pts[:,2].max():.4f}]  (expected within [0, {d_demo}])")

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(projection='3d')
for p1, p2 in zip(p1_demo, p2_demo):
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color='steelblue', linewidth=1, alpha=0.6)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_zlim(0, 1)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title(f'150 sticks confined to a slab of thickness d={d_demo} (drawn inside the unit cube for scale)')
plt.tight_layout()
plt.show()
exp_manager.save_figure(fig, "thin_slab_sanity_check.png")


box_dims = (1.0, 1.0, 0.1)
  x range: [0.0000, 1.0000]  (expected within [0, 1])
  y range: [0.0000, 1.0000]  (expected within [0, 1])
  z range: [-0.0000, 0.1000]  (expected within [0, 0.1])


  [OK] Figure saved: thin_slab_sanity_check.png


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_38694/1335955615.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_872/thin_slab_sanity_check.png'

# Part A: R_eff(d) at fixed (N, a)

---
## 4. Sweep configuration

N = 70 and a = 0.6, the baseline of `percolation_3d.ipynb`, swept from a thin slab (d = 0.1) to the unit cube (d = 1). `contact_distance = 0.03` stays well below the smallest d, so contacts reflect the network and not a slab thinner than the contact radius.

In [5]:
N_fixed = 70
a_fixed = 0.6
contact_distance_fixed = 0.03
d_values = [0.1, 0.15, 0.2, 0.3, 0.5, 0.7, 1.0]
n_trials = 12
R0 = 1000.0
max_attempts = 400
BOUNDARY = "clip"   # generator boundary rule since 2026-09-15 ('reject' before); recorded in every summary

print(f"N={N_fixed}, a={a_fixed}, contact_distance={contact_distance_fixed}")
print(f"d values: {d_values}")
print(f"{n_trials} percolating trials per d")


N=70, a=0.6, contact_distance=0.03
d values: [0.1, 0.15, 0.2, 0.3, 0.5, 0.7, 1.0]
12 percolating trials per d


---
## 5. Run the sweep

For each d, draw networks until `n_trials` of them percolate and record the success rate. Each d reseeds NumPy with `stable_seed(...)`.

In [6]:
results_d = {}       # d -> list of R_eff (distance-dependent), one per percolating trial
perc_rate_d = {}      # d -> fraction of raw draws that percolated
sample_networks_d = {}  # d -> first percolating network's full result dict (for plotting)

for d in d_values:
    # stable_seed(), not hash() -- hash() of a tuple containing round(d, 3)
    # alongside N_fixed/a_fixed is fine (numeric-only), but the two seeded
    # loops below this one in the notebook DO mix in a string tag, and
    # hash() of a string-tagged tuple is randomized per-process
    # (PYTHONHASHSEED). Using stable_seed() everywhere in this notebook
    # keeps all three seed sites on the same reproducible footing. See
    # utils/pipeline_utils.py's stable_seed() docstring.
    seed_base = stable_seed(N_fixed, round(a_fixed, 3), round(d, 3))
    np.random.seed(seed_base)

    R_effs = []
    total_attempts = 0
    while len(R_effs) < n_trials:
        res, attempts = generate_percolating_network_3d(
            N=N_fixed, a=a_fixed, contact_distance=contact_distance_fixed,
            box_dims=(1.0, 1.0, d), k_uniform=R0, max_attempts=max_attempts, boundary=BOUNDARY
        )
        total_attempts += attempts
        if res is None:
            print(f"  [WARNING] d={d}: gave up after {total_attempts} draws, "
                  f"only found {len(R_effs)}/{n_trials} percolating networks")
            break
        if d not in sample_networks_d:
            sample_networks_d[d] = res
        rl = distance_dependent_impedance(res, R0)
        R_effs.append(rl['impedance'])

    rate = len(R_effs) / total_attempts if total_attempts else 0.0
    results_d[d] = R_effs
    perc_rate_d[d] = rate
    print(f"d={d:.2f}   {len(R_effs):2d}/{n_trials} percolating   "
          f"total_draws={total_attempts:4d}   perc_rate={rate:6.1%}")


d=0.10   12/12 percolating   total_draws=  12   perc_rate=100.0%


d=0.15   12/12 percolating   total_draws=  12   perc_rate=100.0%


d=0.20   12/12 percolating   total_draws=  13   perc_rate= 92.3%


d=0.30   12/12 percolating   total_draws=  15   perc_rate= 80.0%


d=0.50   12/12 percolating   total_draws=  18   perc_rate= 66.7%


d=0.70   12/12 percolating   total_draws=  37   perc_rate= 32.4%


d=1.00   12/12 percolating   total_draws= 172   perc_rate=  7.0%


---
## 6. Summary table

In [7]:
header = f"{'d':>6} {'n':>4} {'mean(ohm)':>11} {'median(ohm)':>13} {'std(ohm)':>10} {'perc.rate':>10}"
print(header)
print("-" * len(header))
for d in d_values:
    vals = np.array(results_d[d])
    rate = perc_rate_d[d]
    print(f"{d:6.2f} {len(vals):4d} {vals.mean():11.1f} {np.median(vals):13.1f} "
          f"{vals.std():10.1f} {rate:10.1%}")


     d    n   mean(ohm)   median(ohm)   std(ohm)  perc.rate
-----------------------------------------------------------
  0.10   12       202.8         196.1       67.2     100.0%
  0.15   12       281.2         242.7      114.5     100.0%
  0.20   12       358.4         280.6      224.1      92.3%
  0.30   12       552.1         477.5      326.6      80.0%
  0.50   12       823.7         626.6      433.2      66.7%
  0.70   12      1235.6        1304.5      385.3      32.4%
  1.00   12      1489.8        1389.4      472.6       7.0%


---
## 7. R_eff vs d: box plot and percolation rate

Left: distribution of $R_{eff}$ at each d (log scale), with the mean in red. Right: percolation success rate.

In [8]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

data = [results_d[d] for d in d_values]
ax1.boxplot(data, positions=range(len(d_values)), widths=0.6)
means = [np.mean(results_d[d]) for d in d_values]
ax1.plot(range(len(d_values)), means, 'o-', color='crimson', linewidth=1.5, markersize=5, label='mean')
ax1.set_xticks(range(len(d_values)))
ax1.set_xticklabels(d_values)
ax1.set_yscale('log')
ax1.set_xlabel('d (film thickness)')
ax1.set_ylabel('R_eff (ohm, distance-dependent model, log scale)')
ax1.set_title(f'R_eff vs d  (N={N_fixed}, a={a_fixed}, {n_trials} percolating trials each)')
ax1.grid(True, alpha=0.3, which='both')
ax1.legend()

rates = [perc_rate_d[d] for d in d_values]
ax2.plot(d_values, rates, 'o-', color='darkorange', linewidth=1.5, markersize=6)
ax2.set_xlabel('d (film thickness)')
ax2.set_ylabel('raw percolation success rate')
ax2.set_title('Percolation rate vs d')
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
exp_manager.save_figure(fig, "box_R_eff_vs_d.png")


  [OK] Figure saved: box_R_eff_vs_d.png


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_38694/1821361615.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_872/box_R_eff_vs_d.png'

# Part B: robustness and representative networks

---
## 8. Robustness across (N, a)

The same d-sweep at three (N, a) baselines, with `n_trials_robust = 8` networks per point.

In [9]:
robust_configs = [(70, 0.6), (100, 0.5), (55, 0.8)]
n_trials_robust = 8

results_robust = {}  # (N, a) -> {d: [R_eff, ...]}

for (N, a) in robust_configs:
    results_robust[(N, a)] = {}
    for d in d_values:
        seed_base = stable_seed(N, round(a, 3), round(d, 3), 'robust')
        np.random.seed(seed_base)
        R_effs = []
        total_attempts = 0
        while len(R_effs) < n_trials_robust:
            res, attempts = generate_percolating_network_3d(
                N=N, a=a, contact_distance=contact_distance_fixed,
                box_dims=(1.0, 1.0, d), k_uniform=R0, max_attempts=max_attempts, boundary=BOUNDARY
            )
            total_attempts += attempts
            if res is None:
                print(f"  [WARNING] N={N} a={a} d={d}: gave up after {total_attempts} draws")
                break
            rl = distance_dependent_impedance(res, R0)
            R_effs.append(rl['impedance'])
        results_robust[(N, a)][d] = R_effs
    print(f"N={N:3d} a={a:.1f}: done")


N= 70 a=0.6: done


N=100 a=0.5: done


N= 55 a=0.8: done


In [10]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(robust_configs)))

for (N, a), color in zip(robust_configs, colors):
    means = [np.mean(results_robust[(N, a)][d]) if results_robust[(N, a)][d] else np.nan for d in d_values]
    ax.plot(d_values, means, 'o-', color=color, linewidth=1.8, markersize=5, label=f'N={N}, a={a}')

ax.set_xlabel('d (film thickness)')
ax.set_ylabel('mean R_eff (ohm, log scale)')
ax.set_yscale('log')
ax.set_title('R_eff vs d across several (N, a) baselines')
ax.grid(True, alpha=0.3, which='both')
ax.legend()
plt.tight_layout()
plt.show()
exp_manager.save_figure(fig, "robustness_R_eff_vs_d.png")


  [OK] Figure saved: robustness_R_eff_vs_d.png


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_38694/4252722500.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_872/robustness_R_eff_vs_d.png'

---
## 9. Representative networks: thin slab vs. unit cube

The first percolating N = 70, a = 0.6 network found in Section 5 at the thinnest and at the thickest d, with edges coloured by distance-dependent resistance.

In [11]:
d_thin, d_thick = d_values[0], d_values[-1]

for d in (d_thin, d_thick):
    res = sample_networks_d[d]
    rl = distance_dependent_impedance(res, R0)
    fig = plot_network_3d_colored_by_resistance(
        res['merged_intersections'], res['merged_segment_to_nodes'],
        rl['edge_resistances'],
        title=f"N={N_fixed}, a={a_fixed}, d={d}  ->  R_eff = {rl['impedance']:.0f} \u03a9 (distance-dependent)"
    )
    plt.show()
    exp_manager.save_figure(fig, f"network_d{d}.png")
    exp_manager.save_network(N_fixed, a_fixed, res['points_1'], res['points_2'], res['directions'],
                             filename=f"network_d{d}.json")


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_38694/2158114510.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  [OK] Figure saved: network_d0.1.png
  [OK] JSON saved: network_d0.1.json
  [OK] Figure saved: network_d1.0.png
  [OK] JSON saved: network_d1.0.json


---
## 10. Notes

- In Part A, mean $R_{eff}$ rises monotonically from 203 Ω at d = 0.1 to 1490 Ω at d = 1, and the success rate falls from 100% to 7% (Section 6). With N fixed, the stick density N / d drops as d grows, leaving fewer contacts and longer current paths.
- Section 8 shows the same overall rise at all three baselines, but with 8 networks per point the curves are not monotone point by point (N = 70, a = 0.6 peaks at d = 0.5; N = 55, a = 0.8 dips at d = 0.3).
- d = 1 is the isotropic cube of `percolation_3d.ipynb`. Small d moves toward the thin-sheet picture of `effective_resistance_vs_N_a.ipynb` without reproducing it.
- Fixed areal density (N ∝ d, more material in a thicker film) is expected to reverse the trend; that coupled sweep is not attempted here.
- `contact_distance` is fixed; scaling it with d or with a stick radius is an unexplored parameter.

---
## 11. Save consolidated summary (Part A + robustness)

In [12]:
sweep_records = []
for d in d_values:
    vals = results_d[d]
    sweep_records.append({
        "N": N_fixed, "a": a_fixed, "d": d, "n": len(vals),
        "mean_ohm": float(np.mean(vals)) if vals else None,
        "median_ohm": float(np.median(vals)) if vals else None,
        "std_ohm": float(np.std(vals)) if vals else None,
        "perc_rate": perc_rate_d[d],
        "R_eff_values": vals,
    })

robust_records = []
for (N, a) in robust_configs:
    for d in d_values:
        vals = results_robust[(N, a)][d]
        robust_records.append({
            "N": N, "a": a, "d": d, "n": len(vals),
            "mean_ohm": float(np.mean(vals)) if vals else None,
            "R_eff_values": vals,
        })

summary = {
    "experiment": exp_manager.get_info(),
    "config": {"boundary": BOUNDARY,
        "N_fixed": N_fixed, "a_fixed": a_fixed, "contact_distance": contact_distance_fixed,
        "d_values": d_values, "n_trials": n_trials, "R0": R0, "max_attempts": max_attempts,
        "box_dims_convention": "(1.0, 1.0, d) -- N held fixed as d varies (see section 10)",
    },
    "sweep": sweep_records,
    "robustness_configs": robust_configs,
    "robustness_sweep": robust_records,
}

exp_manager.save_json(summary, "effective_resistance_vs_d_summary.json")

print(f"\nAll results saved to: {exp_manager.experiment_dir}")
for f in sorted(os.listdir(exp_manager.experiment_dir)):
    print(f"  - {f}")


  [OK] JSON saved: effective_resistance_vs_d_summary.json

All results saved to: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_872
  - box_R_eff_vs_d.png
  - effective_resistance_vs_d_summary.json
  - network_d0.1.json
  - network_d0.1.png
  - network_d1.0.json
  - network_d1.0.png
  - robustness_R_eff_vs_d.png
  - thin_slab_sanity_check.png


# Part C: extended (N, a, d) grid

---
## 12. Sweep configuration

A grid over all three parameters, the 3D analogue of Part B of `effective_resistance_vs_N_a.ipynb`. The dense $O(n^3)$ solve again limits the grid, and node counts now grow with N and a and also as d shrinks; an uncapped probe at N = 150, a = 1.0, d = 0.1 did not finish within several minutes locally. `n_max_for_a_d(a, d)` therefore caps N with a node-count model fitted to benchmark runs of `generate_percolating_network_3d`:

$$n_{nodes} \approx 0.0016 \, N^{2.66} a^{2.05} d^{-0.60}$$

For each a, the cap is evaluated at the smallest d and applied to every d at that a.

`LOCAL_PREVIEW = True` runs a small subset locally. `False`, used in the stored run, selects the full grid of 420 combinations with 24 percolating trials each, matching `cluster/run_effective_resistance_d_task.py`.

In [13]:
import time

def n_max_for_a_d(a, d, node_cap=1500, N_ceiling=150, N_floor=20, round_to=5):
    """
    Largest N such that the fitted node-count estimate for (N, a, d) stays
    under node_cap. Mirrors effective_resistance_vs_N_a.ipynb's
    n_max_for_a(a) exactly in spirit (keep the dense O(n_nodes^3) solve
    affordable), generalized to also depend on film thickness d, since a
    thinner box packs the same N sticks more densely (more contacts, more
    nodes) -- see the markdown above for the fitted formula and how it was
    calibrated.
    """
    C, pN, pa, pd = 0.0016, 2.66, 2.05, -0.60
    raw = (node_cap / (C * a ** pa * d ** pd)) ** (1 / pN)
    capped = min(N_ceiling, raw)
    return int(max(N_floor, round(capped / round_to) * round_to))


LOCAL_PREVIEW = False  # False: full 420-combo grid, meant for Apuana (matches
                       # run_effective_resistance_d_task.py's hardcoded grid
                       # exactly -- required for Section 13's cache check to
                       # even look for the array's chunk results). Set back to
                       # True only for a quick local sanity-check run.

if LOCAL_PREVIEW:
    a_values_ext = [0.3, 0.6, 0.9]
    n_per_a = 3
    n_trials_ext = 4
else:
    a_values_ext = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1]  # same a-resolution as effective_resistance_vs_N_a.ipynb
    n_per_a = 6
    n_trials_ext = 24   # 6 until 2026-09-15; must match cluster/run_effective_resistance_d_task.py

d_values_ext = d_values  # reuse the same 7 thickness values as Part A
contact_distance_ext = 0.03
max_attempts_ext = 500
N_floor_ext = 20

grid_ext = {}  # a -> list of N values (adaptive cap, evaluated at the smallest/most demanding d)
for a in a_values_ext:
    n_max = n_max_for_a_d(a, min(d_values_ext))
    Ns = sorted(set(int(round(x / 5) * 5) for x in np.linspace(N_floor_ext, n_max, n_per_a)))
    grid_ext[a] = Ns

combos_ext = [(N, a, d) for a in a_values_ext for N in grid_ext[a] for d in d_values_ext]

exp_manager_ext = setup_experiment()

print(f"LOCAL_PREVIEW = {LOCAL_PREVIEW}")
for a in a_values_ext:
    print(f"  a={a:.1f}:  N_max={n_max_for_a_d(a, min(d_values_ext)):3d}   N values = {grid_ext[a]}")
print(f"\n{len(combos_ext)} (N, a, d) combinations total, {n_trials_ext} percolating trials each")


EXPERIMENT CONFIGURATION
  Environment: Local
  Experiment #: 873
  Timestamp: 20260915_202920
  Results directory: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_873

LOCAL_PREVIEW = False
  a=0.2:  N_max=150   N values = [20, 45, 70, 100, 125, 150]
  a=0.3:  N_max=150   N values = [20, 45, 70, 100, 125, 150]
  a=0.4:  N_max=150   N values = [20, 45, 70, 100, 125, 150]
  a=0.5:  N_max=150   N values = [20, 45, 70, 100, 125, 150]
  a=0.6:  N_max=150   N values = [20, 45, 70, 100, 125, 150]
  a=0.7:  N_max=140   N values = [20, 45, 70, 90, 115, 140]
  a=0.8:  N_max=125   N values = [20, 40, 60, 85, 105, 125]
  a=0.9:  N_max=115   N values = [20, 40, 60, 75, 95, 115]
  a=1.0:  N_max=105   N values = [20, 35, 55, 70, 90, 105]
  a=1.1:  N_max= 95   N values = [20, 35, 50, 65, 80, 95]

420 (N, a, d) combinations total, 24 percolating trials each


**Seeds.** Every seed in this notebook comes from `pipeline_utils.stable_seed(...)`. Earlier versions used `hash((..., 'tag'))`; Python randomizes string hashes per process, so those seeds changed on every run and across SLURM array tasks. Results computed that way are valid random samples but cannot be regenerated network by network.

---
## 13. Run the extended sweep

With `LOCAL_PREVIEW = False`, if all 20 chunk files from the SLURM array job (`cluster/run_effective_resistance_d_array.sbatch` with `run_effective_resistance_d_task.py`) are present in `cluster/effective_resistance_d_results/`, they are loaded instead of recomputed; the stored run loaded 7800 values over 420 combinations. Otherwise the grid runs serially with the procedure of Section 5 and checkpoints after every combination.

In [14]:
cache_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "cluster", "effective_resistance_d_results"))
N_TASKS_ARRAY = 20  # must match run_effective_resistance_d_task.py
chunk_files = [os.path.join(cache_dir, f"chunk_{i:02d}.json") for i in range(N_TASKS_ARRAY)]
use_cache = (not LOCAL_PREVIEW) and all(os.path.exists(p) for p in chunk_files)

results_grid_ext = {}
perc_rate_grid_ext = {}

if use_cache:
    print(f"Found all {N_TASKS_ARRAY} chunk results cached in {cache_dir} "
          f"(from the parallel SLURM array run) -- loading instead of recomputing.")
    for p in chunk_files:
        with open(p) as f:
            rows = json.load(f)
        for row in rows:
            key = (row["N"], row["a"], row["d"])
            results_grid_ext[key] = row["R_eff_values"]
            perc_rate_grid_ext[key] = row["perc_rate"]
    missing = [k for k in combos_ext if k not in results_grid_ext]
    if missing:
        print(f"  [WARNING] {len(missing)} combos from combos_ext have no cached row (grid mismatch?): {missing[:5]}...")
    print(f"Loaded {sum(len(v) for v in results_grid_ext.values())} R_eff values across {len(results_grid_ext)} combinations.")
else:
    if os.path.isdir(cache_dir):
        n_found = sum(os.path.exists(p) for p in chunk_files)
        print(f"Cache dir exists but incomplete ({n_found}/{N_TASKS_ARRAY} chunks) or LOCAL_PREVIEW=True -- recomputing serially.")

    t_start = time.time()
    for idx, (N, a, d) in enumerate(combos_ext, start=1):
        # stable_seed(), not hash() -- see utils/pipeline_utils.py's docstring:
        # hash() of this string-tagged tuple was randomized per-process
        # (PYTHONHASHSEED), silently breaking reproducibility across every
        # separate run of this cell (and across the 20 parallel SLURM array
        # tasks run_effective_resistance_d_task.py splits this same grid
        # into -- each is its own process).
        seed_base = stable_seed(N, round(a, 3), round(d, 3), 'ext')
        np.random.seed(seed_base)

        R_effs = []
        total_attempts = 0
        while len(R_effs) < n_trials_ext:
            res, attempts = generate_percolating_network_3d(
                N=N, a=a, contact_distance=contact_distance_ext,
                box_dims=(1.0, 1.0, d), k_uniform=R0, max_attempts=max_attempts_ext, boundary=BOUNDARY
            )
            total_attempts += attempts
            if res is None:
                print(f"  [WARNING] N={N} a={a} d={d}: gave up after {total_attempts} draws, "
                      f"only found {len(R_effs)}/{n_trials_ext} percolating networks")
                break
            rl = distance_dependent_impedance(res, R0)
            R_effs.append(rl['impedance'])

        rate = len(R_effs) / total_attempts if total_attempts else 0.0
        results_grid_ext[(N, a, d)] = R_effs
        perc_rate_grid_ext[(N, a, d)] = rate

        elapsed = time.time() - t_start
        print(f"[{idx:3d}/{len(combos_ext)}]  N={N:3d} a={a:.1f} d={d:.2f}   "
              f"{len(R_effs)}/{n_trials_ext} percolating   perc_rate={rate:6.1%}   elapsed={elapsed:6.1f}s")

        checkpoint = {
            "done": idx, "total": len(combos_ext), "elapsed_s": elapsed,
            "results_so_far": {f"{N}_{a}_{d}": vals for (N, a, d), vals in results_grid_ext.items()},
            "perc_rate_so_far": {f"{N}_{a}_{d}": r for (N, a, d), r in perc_rate_grid_ext.items()},
        }
        exp_manager_ext.save_json(checkpoint, "extended_sweep_CHECKPOINT.json")

    print(f"\nExtended sweep done in {time.time() - t_start:.1f}s")


Found all 20 chunk results cached in /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/cluster/effective_resistance_d_results (from the parallel SLURM array run) -- loading instead of recomputing.
Loaded 7800 R_eff values across 420 combinations.


---
## 14. Extended-grid plots

One panel per a and one line per N at that a. The grid is ragged, so each panel uses its own N list from `grid_ext[a]`.

In [15]:
fig, axes = plt.subplots(1, len(a_values_ext), figsize=(5.5 * len(a_values_ext), 5), sharey=True)
if len(a_values_ext) == 1:
    axes = [axes]

for ax, a in zip(axes, a_values_ext):
    Ns = grid_ext[a]
    colors_N = plt.cm.plasma(np.linspace(0.15, 0.85, len(Ns)))
    for N, color in zip(Ns, colors_N):
        means = [np.mean(results_grid_ext[(N, a, d)]) if results_grid_ext[(N, a, d)] else np.nan
                 for d in d_values_ext]
        ax.plot(d_values_ext, means, 'o-', color=color, linewidth=1.6, markersize=4, label=f'N={N}')
    ax.set_yscale('log')
    ax.set_xlabel('d (film thickness)')
    ax.set_title(f'a = {a}')
    ax.grid(True, alpha=0.3, which='both')
    ax.legend(fontsize=8)

axes[0].set_ylabel('mean R_eff (ohm, log scale)')
plt.tight_layout()
# No fig.suptitle here: these figures go into the dissertation, where the
# LaTeX \caption carries the title. An embedded title would duplicate it and
# waste vertical space once the figure is scaled to the text width. The
# per-panel titles below are kept, since they label data (the value of a or
# N for that panel), not the figure.
plt.show()
exp_manager_ext.save_figure(fig, "extended_R_eff_vs_d_by_a.png")


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_38694/476540339.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  [OK] Figure saved: extended_R_eff_vs_d_by_a.png


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_873/extended_R_eff_vs_d_by_a.png'

In [16]:
fig, ax = plt.subplots(figsize=(8, 5.5))

Ns_all = np.array([N for (N, a, d) in combos_ext])
as_all = np.array([a for (N, a, d) in combos_ext])
ds_all = np.array([d for (N, a, d) in combos_ext])
means_all = np.array([np.mean(results_grid_ext[k]) if results_grid_ext[k] else np.nan for k in combos_ext])

sc = ax.scatter(ds_all, means_all, c=Ns_all, cmap='viridis', s=40, edgecolor='k', linewidth=0.3)
ax.set_yscale('log')
ax.set_xlabel('d (film thickness)')
ax.set_ylabel('mean R_eff (ohm, log scale)')
ax.set_title('Extended grid: every (N, a, d) point, colored by N')
ax.grid(True, alpha=0.3, which='both')
fig.colorbar(sc, ax=ax, label='N')
plt.tight_layout()
plt.show()
exp_manager_ext.save_figure(fig, "extended_scatter_R_eff_vs_d.png")


  [OK] Figure saved: extended_scatter_R_eff_vs_d.png


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_38694/2114013311.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_873/extended_scatter_R_eff_vs_d.png'

---
## 15. Power-law fit and Part C summary

### C.1 Three-parameter power law over the whole grid

The scalar fit the thesis quotes for the 3D grid: the general (N, a) fit of `inverse_N_a_from_R_eff.ipynb` extended with a $\log_{10} d$ term. It is fitted by linear least squares in log space to every percolating realization, not to per-combination means, so combinations with more realizations weigh more. Standard errors come from the OLS covariance, and the $R^2$ without the d term is printed so the contribution of thickness is explicit.

Stored fit: $R_{eff} \approx 8.64 \times 10^5 \, N^{-1.892} a^{-2.634} d^{0.804}$ with $R^2 = 0.854$ (0.635 without the d term), over 7800 realizations in 341 combinations. The cell after the fit saves the Part C summary.

In [17]:
X_rows, y_rows = [], []
for (N, a, d), vals in results_grid_ext.items():
    for v in vals:
        X_rows.append([1.0, np.log10(N), np.log10(a), np.log10(d)])
        y_rows.append(np.log10(v))
X3 = np.array(X_rows); y3 = np.array(y_rows)
n_obs, n_par = X3.shape

beta, *_ = np.linalg.lstsq(X3, y3, rcond=None)
pred = X3 @ beta
ss_res = float(np.sum((y3 - pred) ** 2)); ss_tot = float(np.sum((y3 - y3.mean()) ** 2))
r2_3d = 1.0 - ss_res / ss_tot
sigma2 = ss_res / (n_obs - n_par)
se = np.sqrt(np.diag(sigma2 * np.linalg.inv(X3.T @ X3)))

# same fit with the d term dropped, to show what thickness adds
beta_nod, *_ = np.linalg.lstsq(X3[:, :3], y3, rcond=None)
r2_no_d = 1.0 - float(np.sum((y3 - X3[:, :3] @ beta_nod) ** 2)) / ss_tot

fit_3d = {
    "model": "R_eff = A * N^p * a^q * d^s (log10 least squares over individual realizations)",
    "n_realizations": int(n_obs), "n_combinations": sum(1 for v in results_grid_ext.values() if v),
    "A": float(10 ** beta[0]), "p": float(beta[1]), "q": float(beta[2]), "s": float(beta[3]),
    "se_p": float(se[1]), "se_q": float(se[2]), "se_s": float(se[3]),
    "t_s": float(beta[3] / se[3]), "dof": int(n_obs - n_par),
    "r2": float(r2_3d), "r2_without_d": float(r2_no_d),
}
print(f"{n_obs} realizations over {fit_3d['n_combinations']} (N, a, d) combinations with at least one percolating realization")
print(f"R_eff ~= {fit_3d['A']:.3g} * N^{beta[1]:.3f} * a^{beta[2]:.3f} * d^{beta[3]:.3f}    R^2 = {r2_3d:.3f}")
print(f"standard errors: p {se[1]:.3f}, q {se[2]:.3f}, s {se[3]:.3f}   (t for the d exponent: {beta[3]/se[3]:.0f} on {n_obs-n_par} dof)")
print(f"same fit without the d term: R^2 = {r2_no_d:.3f}")

7800 realizations over 341 (N, a, d) combinations with at least one percolating realization
R_eff ~= 8.64e+05 * N^-1.892 * a^-2.634 * d^0.804    R^2 = 0.854
standard errors: p 0.011, q 0.016, s 0.007   (t for the d exponent: 108 on 7796 dof)
same fit without the d term: R^2 = 0.635


In [18]:
sweep_records_ext = []
for (N, a, d) in combos_ext:
    vals = results_grid_ext[(N, a, d)]
    sweep_records_ext.append({
        "N": N, "a": a, "d": d, "n": len(vals),
        "mean_ohm": float(np.mean(vals)) if vals else None,
        "median_ohm": float(np.median(vals)) if vals else None,
        "std_ohm": float(np.std(vals)) if vals else None,
        "perc_rate": perc_rate_grid_ext[(N, a, d)],
        "R_eff_values": vals,
    })

summary_ext = {
    "experiment": exp_manager_ext.get_info(),
    "config": {
        "LOCAL_PREVIEW": LOCAL_PREVIEW,
        "a_values": a_values_ext,
        "grid_ext": {str(a): Ns for a, Ns in grid_ext.items()},
        "d_values": d_values_ext,
        "contact_distance": contact_distance_ext, "n_trials": n_trials_ext, "boundary": BOUNDARY,
        "R0": R0, "max_attempts": max_attempts_ext,
        "grid_design": "adaptive: N_max(a) = n_max_for_a_d(a, d_min) keeps estimated nodes <= node_cap=1500",
    },
    "sweep": sweep_records_ext,
    "fit_3d": fit_3d,
}

exp_manager_ext.save_json(summary_ext, "effective_resistance_vs_d_extended_summary.json")

print(f"\nAll results saved to: {exp_manager_ext.experiment_dir}")
for f in sorted(os.listdir(exp_manager_ext.experiment_dir)):
    print(f"  - {f}")


  [OK] JSON saved: effective_resistance_vs_d_extended_summary.json

All results saved to: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_873
  - effective_resistance_vs_d_extended_summary.json
  - extended_R_eff_vs_d_by_a.png
  - extended_scatter_R_eff_vs_d.png


---
## 16. Running the full grid on Apuana

The stored run used `LOCAL_PREVIEW = False` and loaded the cached chunks. To regenerate the 420-combination grid:

**Option A, parallel (recommended).** `cluster/run_effective_resistance_d_array.sbatch` with `run_effective_resistance_d_task.py` splits the grid into 20 SLURM array tasks (at most 4 at a time under the `simple` QOS). Each task writes `cluster/effective_resistance_d_results/chunk_NN.json`; once all 20 exist, Section 13 loads them.

```bash
cd ~/ColabProject/cluster
mkdir -p logs
sbatch run_effective_resistance_d_array.sbatch
```

**Option B, serial.** `cluster/run_effective_resistance_d_sweep.sbatch` executes this whole notebook with `nbclient` as one job. It is simpler but does not parallelize Section 13, so it takes about 20 times longer than one array task.

```bash
cd ~/ColabProject/cluster
mkdir -p logs
sbatch run_effective_resistance_d_sweep.sbatch
```

The array task script always uses the full grid; `LOCAL_PREVIEW` only affects the notebook's own loop. The serial branch writes `extended_sweep_CHECKPOINT.json` after every combination but restarts from the first combination when rerun. Array tasks are independent, so only failed chunks need resubmitting.